In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
df_user=session.sql("SELECT * FROM tbl_yelp_user").to_pandas()


In [ ]:
df_user.head()

In [ ]:
df_user.describe()

In [ ]:
df_user.info()

In [ ]:
df_user["YELPING_SINCE"].min()

In [ ]:
(df_user["YELPING_SINCE"] == pd.Timestamp(0)).sum() 

In [ ]:
(df_user["USER_ID"].str.strip() == "").sum()


In [ ]:
df_user.isnull().sum()

In [ ]:
(df_user["ELITE"].str.strip() == "").sum()


In [ ]:
df_user["ELITE"] = df_user["ELITE"].fillna("NOT ELITE")


In [ ]:
df_user["ELITE"].isna().sum()


In [ ]:
df_user.shape

In [ ]:
df_user[df_user.duplicated()]

In [ ]:
df=df_user.copy()

In [ ]:
df["account_age_years"] = (pd.Timestamp.now() - df_user["YELPING_SINCE"]).dt.total_seconds()/(60*60*24*365)


In [ ]:
df["account_age_years"].describe()


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df["account_age_years"].value_counts().head(10)

In [ ]:
df["YELPING_SINCE"].value_counts().sort_index().cumsum().plot(kind="line")
plt.title("Cumulative Growth of Yelp Users Over Time")
plt.xlabel("Year")
plt.ylabel("Total Users (Cumulative)")
plt.show()

In [ ]:
df.head(10)

In [ ]:
df_user.head(10)

In [ ]:
df["JOIN_YEAR"] = df["YELPING_SINCE"].dt.year
df["JOIN_YEAR"].value_counts().sort_index().plot(kind="bar")
plt.title("New Users Joined Per Year")
plt.xlabel("Year")
plt.ylabel("Users")
plt.show()

In [ ]:
df["ELITE_FLAG"] = df["ELITE"].apply(lambda x: 0 if x in ["", " ", "NOT ELITE"] else 1)

elite_avg = df.groupby("ELITE_FLAG")["REVIEW_COUNT"].mean()
elite_avg.plot(kind="bar")
plt.title("Avg Reviews: Elite vs Non-Elite Users")
plt.xlabel("0 = Non Elite, 1 = Elite")
plt.ylabel("Avg Review Count")
plt.show()


In [ ]:
corr_df = df[["REVIEW_COUNT", "JOIN_YEAR", "account_age_years", "ELITE_FLAG"]].corr()
import numpy as np
plt.imshow(corr_df)
plt.title("Correlation Matrix: User Engagement & Longevity")
plt.xticks(range(len(corr_df)), corr_df.columns)
plt.yticks(range(len(corr_df)), corr_df.columns)
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sort users by review activity
sorted_counts = df["REVIEW_COUNT"].sort_values(ascending=False).values
total_reviews = sorted_counts.sum()

# Compute cumulative % contribution
cumulative_percent = np.cumsum(sorted_counts) / total_reviews * 100

# Plot Pareto curve
plt.plot(range(len(cumulative_percent)), cumulative_percent)
plt.axhline(80, linestyle="--")  # 80% reference line
plt.title("Pareto Curve: % Contribution of Reviews by Most Active Users")
plt.xlabel("Users (sorted by activity rank)")
plt.ylabel("Cumulative % of Reviews")
plt.show()

In [ ]:
df["ELITE_FLAG"] = df["ELITE"].apply(lambda x: 0 if x in ["", " ", "NOT ELITE"] else 1)


In [ ]:
cohort = df.groupby("JOIN_YEAR")["REVIEW_COUNT"].mean()
cohort.plot(kind="line")
plt.title("Average Reviews by User Signup Year (Cohort Engagement)")
plt.xlabel("Signup Year")
plt.ylabel("Avg Review Count")
plt.show()

In [ ]:
df["TIER"] = pd.qcut(df_user["REVIEW_COUNT"], q=3, labels=["Light", "Medium", "Heavy"])


In [ ]:
df.groupby("TIER")["REVIEW_COUNT"].mean().plot(kind="bar")
plt.title("Average Reviews per User Engagement Tier")
plt.xlabel("User Tier")
plt.ylabel("Avg Review Count")
plt.show()

In [ ]:
# Convert to string just to be safe
df["ELITE"] = df["ELITE"].astype(str)

# Replace placeholders into real NOT ELITE
df["ELITE"] = df["ELITE"].replace(["nan","None"," ",""], "NOT ELITE")

# Extract years into list
df["ELITE_YEARS"] = df["ELITE"].apply(
    lambda x: [] if x == "NOT ELITE" else [int(y) for y in x.split(",") if y.isdigit()]
)

# Explode so each elite year becomes its own row
df_elite = df.explode("ELITE_YEARS")

# Count elite users per year
elite_counts = df_elite.groupby("ELITE_YEARS")["USER_ID"].nunique()

# Plot only years where elite existed
elite_counts.sort_index().plot(kind="bar")
plt.title("Number of Elite Users Per Year")
plt.xlabel("Year")
plt.ylabel("Elite Users")
plt.show()

print(elite_counts)


In [ ]:
df_checkin=session.sql("SELECT * FROM tbl_checkins;").to_pandas()


In [ ]:
df_checkin.head()

In [ ]:
df_checkin.info()

In [ ]:
df_checkin.describe()

In [ ]:
(df_checkin["BUSINESS_ID"].str.strip() == " ").sum()


In [ ]:
df_checkin["DATE_VISITED"].isna().sum()
df_checkin["BUSINESS_ID"].isna().sum()


In [ ]:
print(df_checkin.duplicated().sum(), "fully duplicate rows")


In [ ]:
duplicates = df_checkin[df_checkin.duplicated(keep=False)]
print(duplicates)

In [ ]:
df_checkin = df_checkin.drop_duplicates()



In [ ]:
df_check=df_checkin.copy()

In [ ]:
print(df_checkin.duplicated().sum(), "fully duplicate rows")


In [ ]:
print(df_check.duplicated().sum(), "fully duplicate rows")


In [ ]:
SELECT
  COUNT(*) AS total_rows,
  COUNT_IF(business_id IS NULL OR TRIM(business_id)='') AS business_id_missing,
  COUNT_IF(date_visited IS NULL) AS date_missing
FROM tbl_checkins;

In [ ]:
df_check.groupby("BUSINESS_ID").size().describe()


In [ ]:
# Monthly check-in volume trend
df_check["month"] = df_checkin["DATE_VISITED"].dt.to_period("M")
df_check.groupby("month").size().sort_index().tail(12)

In [ ]:
df_check["MONTH_ONLY"] = df_check["DATE_VISITED"].dt.month

# Group by month number and plot
df_check.groupby("MONTH_ONLY").size().sort_index().plot()
plt.title("Check-ins Trend (Month Wise)")
plt.xlabel("Month (1-12)")
plt.ylabel("Check-ins")
plt.xticks(range(1,13))  # forces x-axis to show only 1 to 12
plt.show()

In [ ]:
df_checkin.groupby("BUSINESS_ID").size().sort_values(ascending=False).head(10)


In [ ]:
df_check.head()

In [ ]:
df.info()

In [ ]:
df_user.info()

In [ ]:
df_check.info()
df_checkin.info()

In [ ]:
df_check["YEAR"] = df_check["DATE_VISITED"].dt.year

In [ ]:
df_check.groupby(["BUSINESS_ID","YEAR"]).size().reset_index(name="CHECKINS")


In [ ]:
df_check.groupby("BUSINESS_ID")["DATE_VISITED"].max().count()


In [ ]:
df_check.count()

In [ ]:
# 4. Average check-ins per year (overall platform maturity)
avg_year = df_check.groupby("YEAR").size() / df_check["BUSINESS_ID"].nunique()
avg_year.plot(kind="line")
plt.title("Avg Check-ins Per Business by Year")
plt.xlabel("Year")
plt.ylabel("Avg Check-ins")
plt.show()

In [ ]:
# 5. Venue Pareto (top businesses create most check-ins?)
venue_counts = df_check["BUSINESS_ID"].value_counts()
pareto_venues = venue_counts.cumsum() / venue_counts.sum() * 100
pareto_venues.plot(kind="line")
plt.axhline(80, linestyle="--")
plt.title("Pareto Curve (Venue Visit Concentration)")
plt.xlabel("Businesses sorted by check-ins")
plt.ylabel("Cumulative % of Check-ins")
plt.show()